# Echo-TOF WIFF2 Data Processing & Analysis

이 노트북은 Echo-TOF에서 수집한 WIFF2 파일 데이터를 처리하고 분석하는 워크플로우입니다.

- Step 1: 환경 설정 및 라이브러리 임포트
- Step 2: WIFF2 파일 로드 및 초기 데이터 가시화
- Step 3: 적분 로직 적용 및 결과 가시화
- Step 4: 화합물 구조 입력 및 패턴 가시화

## Step 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# ─── Colab 환경: 필수 패키지 설치 ───
# pyopenms: mzML/WIFF2 파일 처리
# rdkit: 화합물 구조 처리
!pip install -q pyopenms rdkit-pypi pandas matplotlib seaborn numpy scipy

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Mass Spec 데이터 처리
try:
    import pyopenms as oms
    print(f"pyOpenMS version: {oms.__version__}")
    HAS_OPENMS = True
except ImportError:
    print("pyOpenMS not available. Using CSV/mzML fallback mode.")
    HAS_OPENMS = False

# 화합물 구조
try:
    from rdkit import Chem
    from rdkit.Chem import Draw, Descriptors, AllChem
    print(f"RDKit available")
    HAS_RDKIT = True
except ImportError:
    print("RDKit not available. Structure features disabled.")
    HAS_RDKIT = False

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

print("\n✅ Environment ready!")

## Step 2. WIFF2 파일 로드 및 초기 데이터 가시화

### 2.1 데이터 로드

WIFF2 파일을 직접 로드하거나, 변환된 mzML/CSV 파일을 사용합니다.

> **Colab에서 WIFF2 사용**: WIFF2는 SCIEX 독점 포맷이므로 ProteoWizard msconvert로 mzML로 변환 후 사용을 권장합니다.
> 또는 SCIEX OS에서 직접 텍스트/CSV로 내보낸 데이터를 사용할 수 있습니다.

In [ ]:
# ─── 데이터 소스 설정 ───
# 옵션 1: Google Drive에서 파일 로드
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = Path('/content/drive/MyDrive/EchoTOF_Data/')

# 옵션 2: 로컬 업로드
# from google.colab import files
# uploaded = files.upload()

# 옵션 3: 샘플 데이터 생성 (데모용)
DATA_PATH = Path('./data')
DATA_PATH.mkdir(exist_ok=True)

print(f"Data path: {DATA_PATH}")

In [ ]:
# ─── 샘플 데이터 생성 (실제 데이터 없을 때 데모용) ───

def generate_sample_spectrum(mz_center=195.088, n_points=1000, noise_level=50):
    """단일 질량 스펙트럼 시뮬레이션"""
    mz = np.linspace(50, 500, n_points)
    intensity = np.zeros_like(mz)

    # 주요 피크 추가 (Caffeine 예시: 195.088, 138.066, 110.071)
    peaks = [
        (195.088, 10000, 0.02),   # [M+H]+
        (138.066, 4500, 0.02),    # fragment
        (110.071, 2000, 0.02),    # fragment
        (217.070, 1500, 0.02),    # [M+Na]+
        (82.053,  800, 0.02),     # fragment
    ]
    for center, height, width in peaks:
        intensity += height * np.exp(-((mz - center)**2) / (2 * width**2))

    # 노이즈 추가
    intensity += np.abs(np.random.normal(0, noise_level, n_points))
    return mz, intensity

def generate_plate_data(n_wells=96, sm_mw=250.0, product_mw=352.0):
    """96웰 플레이트 시뮬레이션 데이터"""
    rows = 'ABCDEFGH'
    cols = range(1, 13)
    wells = [f"{r}{c}" for r in rows for c in cols]

    np.random.seed(42)
    data = []
    for i, well in enumerate(wells):
        # 랜덤 수율 (일부 웰은 반응 안됨)
        yield_pct = np.clip(np.random.normal(45, 25), 0, 100)
        if np.random.random() < 0.15:  # 15% 실패
            yield_pct = np.random.uniform(0, 5)

        sm_area = 10000 * (1 - yield_pct/100) + np.random.normal(0, 500)
        product_area = 10000 * (yield_pct/100) + np.random.normal(0, 500)
        sm_area = max(0, sm_area)
        product_area = max(0, product_area)

        data.append({
            'well': well,
            'row': well[0],
            'col': int(well[1:]),
            'sm_mz': sm_mw + 1.0073,      # [M+H]+
            'product_mz': product_mw + 1.0073,
            'sm_area': sm_area,
            'product_area': product_area,
            'total_area': sm_area + product_area + np.random.uniform(500, 2000),
            'yield_area_ratio': 0,  # 후에 계산
        })

    df = pd.DataFrame(data)
    df['yield_area_ratio'] = (df['product_area'] / df['total_area'] * 100).clip(0, 100)
    return df

# 생성
sample_mz, sample_int = generate_sample_spectrum()
plate_df = generate_plate_data()

print(f"Sample spectrum: {len(sample_mz)} points, m/z range [{sample_mz[0]:.1f}, {sample_mz[-1]:.1f}]")
print(f"Plate data: {len(plate_df)} wells")
plate_df.head()

### 2.2 전체 이온 크로마토그램 (TIC) & 질량 스펙트럼

In [ ]:
# ─── 질량 스펙트럼 가시화 ───

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 왼쪽: 전체 스펙트럼
ax1 = axes[0]
ax1.plot(sample_mz, sample_int, color='#0078D4', linewidth=0.8)
ax1.set_xlabel('m/z')
ax1.set_ylabel('Intensity')
ax1.set_title('Mass Spectrum (Full Range)')

# 주요 피크 라벨
peak_mzs = [195.088, 138.066, 110.071, 217.070]
for pmz in peak_mzs:
    idx = np.argmin(np.abs(sample_mz - pmz))
    ax1.annotate(f'{pmz:.3f}', xy=(sample_mz[idx], sample_int[idx]),
                 xytext=(0, 10), textcoords='offset points',
                 ha='center', fontsize=8, color='red')

# 오른쪽: 확대 (m/z 180-230)
ax2 = axes[1]
mask = (sample_mz >= 180) & (sample_mz <= 230)
ax2.plot(sample_mz[mask], sample_int[mask], color='#E85D04', linewidth=1.0)
ax2.set_xlabel('m/z')
ax2.set_ylabel('Intensity')
ax2.set_title('Zoomed Region (m/z 180-230)')

plt.tight_layout()
plt.show()

### 2.3 mzML 파일 로드 (실제 데이터용)

```
실제 WIFF2 → mzML 변환 후 사용하는 코드입니다.
데모 모드에서는 스킵됩니다.
```

In [ ]:
# ─── mzML 파일 로드 (실제 데이터용) ───

def load_mzml(filepath):
    """mzML 파일에서 스펙트럼 데이터 추출"""
    if not HAS_OPENMS:
        print("pyOpenMS required. Install: pip install pyopenms")
        return None

    exp = oms.MSExperiment()
    oms.MzMLFile().load(str(filepath), exp)

    spectra = []
    for spec in exp:
        mz, intensity = spec.get_peaks()
        spectra.append({
            'rt': spec.getRT(),
            'ms_level': spec.getMSLevel(),
            'mz': mz,
            'intensity': intensity,
            'tic': np.sum(intensity),
        })

    print(f"Loaded {len(spectra)} spectra from {filepath}")
    print(f"  MS1: {sum(1 for s in spectra if s['ms_level']==1)}")
    print(f"  MS2: {sum(1 for s in spectra if s['ms_level']==2)}")
    print(f"  RT range: {spectra[0]['rt']:.1f} - {spectra[-1]['rt']:.1f} sec")
    return spectra

def extract_eic(spectra, target_mz, tolerance_da=0.01):
    """추출 이온 크로마토그램 (XIC/EIC)"""
    rts, intensities = [], []
    for s in spectra:
        if s['ms_level'] != 1:
            continue
        mask = np.abs(s['mz'] - target_mz) <= tolerance_da
        rts.append(s['rt'])
        intensities.append(np.sum(s['intensity'][mask]) if mask.any() else 0)
    return np.array(rts), np.array(intensities)

def extract_tic(spectra):
    """전체 이온 크로마토그램 (TIC)"""
    rts = [s['rt'] for s in spectra if s['ms_level'] == 1]
    tics = [s['tic'] for s in spectra if s['ms_level'] == 1]
    return np.array(rts), np.array(tics)

# ─── 사용 예시 (실제 mzML 파일이 있을 때) ───
# spectra = load_mzml('/content/drive/MyDrive/data/sample.mzML')
# rt_tic, tic = extract_tic(spectra)
# rt_eic, eic = extract_eic(spectra, target_mz=195.088)

print("✅ mzML loader functions defined.")
print("   사용법: spectra = load_mzml('your_file.mzML')")

## Step 3. 적분 로직 적용 및 결과 가시화

### 3.1 피크 검출 알고리즘

In [ ]:
# ─── 피크 검출 & 적분 엔진 ───

from scipy.signal import find_peaks, savgol_filter
from scipy.ndimage import uniform_filter1d

class PeakIntegrator:
    """Echo-TOF 스펙트럼 피크 검출 및 적분"""

    def __init__(self, noise_factor=3.0, min_peak_height=None):
        self.noise_factor = noise_factor
        self.min_peak_height = min_peak_height

    def estimate_noise(self, y):
        """MAD 기반 노이즈 추정"""
        diff = np.diff(y)
        return np.median(np.abs(diff)) * 1.4826 / np.sqrt(2)

    def baseline_correct(self, y, window=50):
        """Rolling minimum baseline 보정"""
        from scipy.ndimage import minimum_filter1d
        baseline = minimum_filter1d(y, size=window)
        baseline = uniform_filter1d(baseline, size=window)
        return y - baseline, baseline

    def detect_peaks(self, mz, intensity, prominence_factor=5):
        """피크 검출 + 경계 결정"""
        # Baseline 보정
        corrected, baseline = self.baseline_correct(intensity)

        # 노이즈 추정
        noise = self.estimate_noise(corrected)
        threshold = noise * self.noise_factor

        # 피크 검출
        min_h = self.min_peak_height or threshold
        peaks_idx, properties = find_peaks(
            corrected,
            height=min_h,
            prominence=noise * prominence_factor,
            distance=3,
            width=2,
        )

        results = []
        for i, idx in enumerate(peaks_idx):
            # 피크 경계 (반폭 기준)
            left = int(properties.get('left_ips', [idx-5])[i]) if 'left_ips' in properties else max(0, idx-5)
            right = int(properties.get('right_ips', [idx+5])[i]) if 'right_ips' in properties else min(len(mz)-1, idx+5)

            # 면적 적분 (사다리꼴)
            area = np.trapz(corrected[left:right+1], mz[left:right+1])

            results.append({
                'peak_idx': idx,
                'mz': mz[idx],
                'intensity': intensity[idx],
                'corrected_intensity': corrected[idx],
                'area': max(0, area),
                'sn': corrected[idx] / noise if noise > 0 else 0,
                'left_idx': left,
                'right_idx': right,
                'baseline': baseline[idx],
            })

        return pd.DataFrame(results), corrected, baseline

    def integrate_target(self, mz, intensity, target_mz, tolerance_da=0.05):
        """특정 m/z 타겟의 면적만 적분"""
        peaks_df, corrected, baseline = self.detect_peaks(mz, intensity)
        if peaks_df.empty:
            return 0.0, peaks_df

        # 타겟 m/z에 가장 가까운 피크
        peaks_df['mz_diff'] = np.abs(peaks_df['mz'] - target_mz)
        within_tol = peaks_df[peaks_df['mz_diff'] <= tolerance_da]

        if within_tol.empty:
            return 0.0, peaks_df

        best = within_tol.loc[within_tol['mz_diff'].idxmin()]
        return best['area'], peaks_df

# 인스턴스 생성
integrator = PeakIntegrator(noise_factor=3.0)
print("✅ PeakIntegrator ready.")

In [ ]:
# ─── 피크 검출 실행 & 가시화 ───

peaks_df, corrected, baseline = integrator.detect_peaks(sample_mz, sample_int)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]})

# 상단: 스펙트럼 + 피크 표시
ax1 = axes[0]
ax1.plot(sample_mz, sample_int, color='#888', linewidth=0.5, alpha=0.5, label='Raw')
ax1.plot(sample_mz, corrected, color='#0078D4', linewidth=0.8, label='Baseline corrected')
ax1.plot(sample_mz, baseline, color='orange', linewidth=1, linestyle='--', label='Baseline', alpha=0.7)

# 적분 영역 색칠
for _, peak in peaks_df.iterrows():
    left, right = int(peak['left_idx']), int(peak['right_idx'])
    ax1.fill_between(sample_mz[left:right+1], 0, corrected[left:right+1],
                     alpha=0.3, color='#2196F3')
    ax1.annotate(f"m/z {peak['mz']:.3f}\nArea={peak['area']:.0f}\nS/N={peak['sn']:.1f}",
                xy=(peak['mz'], peak['corrected_intensity']),
                xytext=(0, 15), textcoords='offset points',
                ha='center', fontsize=7, color='red',
                arrowprops=dict(arrowstyle='->', color='red', lw=0.5))

ax1.set_ylabel('Intensity')
ax1.set_title(f'Peak Detection: {len(peaks_df)} peaks found')
ax1.legend(fontsize=8)

# 하단: 피크 결과 바 차트
ax2 = axes[1]
if not peaks_df.empty:
    ax2.bar(peaks_df['mz'], peaks_df['area'], width=1.5, color='#0078D4', alpha=0.7)
    ax2.set_xlabel('m/z')
    ax2.set_ylabel('Peak Area')
    ax2.set_title('Integrated Peak Areas')

plt.tight_layout()
plt.show()

# 결과 테이블
print(f"\n{'='*60}")
print(f"  Detected Peaks: {len(peaks_df)}")
print(f"{'='*60}")
peaks_df[['mz', 'intensity', 'area', 'sn']].round(2)

### 3.2 96웰 플레이트 배치 적분

In [ ]:
# ─── 96웰 플레이트 수율 히트맵 ───

def plot_plate_heatmap(df, value_col='yield_area_ratio', title='96-Well Plate Yield Heatmap'):
    """SCIEX OS 스타일 96웰 히트맵"""
    pivot = df.pivot(index='row', columns='col', values=value_col)

    fig, ax = plt.subplots(figsize=(14, 5))

    # 히트맵 (초록=높은 수율, 빨강=낮은 수율)
    cmap = sns.color_palette("RdYlGn", as_cmap=True)
    hm = sns.heatmap(pivot, annot=True, fmt='.0f', cmap=cmap,
                     vmin=0, vmax=100, linewidths=1, linecolor='white',
                     cbar_kws={'label': 'Yield (%)', 'shrink': 0.8},
                     ax=ax)

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

    # 통계
    mean_yield = df[value_col].mean()
    hit_rate = (df[value_col] > 20).mean() * 100  # 20% 이상 = hit

    ax.text(1.15, 0.5, f"Mean: {mean_yield:.1f}%\nHit rate: {hit_rate:.0f}%\n(>20% cutoff)",
            transform=ax.transAxes, fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

    return pivot

pivot = plot_plate_heatmap(plate_df)

In [ ]:
# ─── 웰별 수율 분포 + Top/Bottom 랭킹 ───

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. 수율 분포 히스토그램
ax1 = axes[0]
ax1.hist(plate_df['yield_area_ratio'], bins=20, color='#0078D4', edgecolor='white', alpha=0.8)
ax1.axvline(plate_df['yield_area_ratio'].mean(), color='red', linestyle='--', label=f"Mean: {plate_df['yield_area_ratio'].mean():.1f}%")
ax1.axvline(20, color='orange', linestyle=':', label='Hit cutoff (20%)')
ax1.set_xlabel('Yield (%)')
ax1.set_ylabel('Count')
ax1.set_title('Yield Distribution')
ax1.legend(fontsize=8)

# 2. SM vs Product 면적 산점도
ax2 = axes[1]
colors = plate_df['yield_area_ratio'].values
sc = ax2.scatter(plate_df['sm_area'], plate_df['product_area'],
                 c=colors, cmap='RdYlGn', vmin=0, vmax=100, s=30, alpha=0.8, edgecolors='#333', linewidths=0.3)
plt.colorbar(sc, ax=ax2, label='Yield (%)', shrink=0.8)
ax2.set_xlabel('SM Area')
ax2.set_ylabel('Product Area')
ax2.set_title('SM vs Product')

# 3. Top 10 / Bottom 10
ax3 = axes[2]
top10 = plate_df.nlargest(10, 'yield_area_ratio')[['well', 'yield_area_ratio']]
bot10 = plate_df.nsmallest(10, 'yield_area_ratio')[['well', 'yield_area_ratio']]
combined = pd.concat([top10, bot10])
colors_bar = ['#2E8B57' if v > 20 else '#DC143C' for v in combined['yield_area_ratio']]
ax3.barh(combined['well'], combined['yield_area_ratio'], color=colors_bar)
ax3.set_xlabel('Yield (%)')
ax3.set_title('Top 10 & Bottom 10 Wells')
ax3.invert_yaxis()

plt.tight_layout()
plt.show()

## Step 4. 화합물 구조 입력 및 패턴 가시화

### 4.1 화합물 정보 입력 (SMILES)

In [ ]:
# ─── 화합물 구조 정의 ───

COMPOUNDS = {
    'Starting Material': {
        'smiles': 'c1ccc(NC(=O)c2ccccc2)cc1',  # Benzanilide (예시)
        'mw': 197.084,
        'role': 'SM',
    },
    'Product': {
        'smiles': 'c1ccc(NC(=O)c2ccc(OC)cc2)cc1',  # Methoxybenzanilide (예시)
        'mw': 227.095,
        'role': 'Product',
    },
    'Caffeine (QC)': {
        'smiles': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
        'mw': 194.080,
        'role': 'IS',  # Internal Standard
    },
}

# 분자량 + m/z 계산
H = 1.00728  # proton mass
for name, info in COMPOUNDS.items():
    info['mz_pos'] = info['mw'] + H      # [M+H]+
    info['mz_neg'] = info['mw'] - H      # [M-H]-
    info['mz_na'] = info['mw'] + 22.9892 # [M+Na]+

# 테이블 출력
comp_df = pd.DataFrame([
    {'Name': name, 'SMILES': info['smiles'][:40], 'MW': f"{info['mw']:.3f}",
     '[M+H]+': f"{info['mz_pos']:.4f}", '[M+Na]+': f"{info['mz_na']:.4f}",
     'Role': info['role']}
    for name, info in COMPOUNDS.items()
])
comp_df

In [ ]:
# ─── 화합물 구조 시각화 (RDKit) ───

if HAS_RDKIT:
    mols = []
    legends = []
    for name, info in COMPOUNDS.items():
        mol = Chem.MolFromSmiles(info['smiles'])
        if mol:
            mols.append(mol)
            legends.append(f"{name}\nMW={info['mw']:.3f}\n[M+H]+={info['mz_pos']:.4f}")

    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(350, 250), legends=legends)
    display(img)
else:
    print("RDKit not available. Structure visualization skipped.")
    for name, info in COMPOUNDS.items():
        print(f"  {name}: {info['smiles']} (MW={info['mw']:.3f})")

### 4.2 화합물-스펙트럼 패턴 매칭

In [ ]:
# ─── 화합물별 m/z 매칭 & 패턴 가시화 ───

def match_compounds_to_peaks(peaks_df, compounds, tolerance_da=0.05):
    """피크를 화합물에 매핑"""
    assignments = []
    for _, peak in peaks_df.iterrows():
        matched = False
        for name, info in compounds.items():
            for adduct, mz_key in [('[M+H]+', 'mz_pos'), ('[M-H]-', 'mz_neg'), ('[M+Na]+', 'mz_na')]:
                if abs(peak['mz'] - info[mz_key]) <= tolerance_da:
                    assignments.append({
                        **peak.to_dict(),
                        'compound': name,
                        'adduct': adduct,
                        'theoretical_mz': info[mz_key],
                        'error_da': peak['mz'] - info[mz_key],
                        'error_ppm': (peak['mz'] - info[mz_key]) / info[mz_key] * 1e6,
                    })
                    matched = True
        if not matched:
            assignments.append({
                **peak.to_dict(),
                'compound': 'Unknown',
                'adduct': '',
                'theoretical_mz': 0,
                'error_da': 0,
                'error_ppm': 0,
            })
    return pd.DataFrame(assignments)

# 매칭 실행
assignments_df = match_compounds_to_peaks(peaks_df, COMPOUNDS, tolerance_da=0.1)

# 가시화
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sample_mz, corrected, color='#888', linewidth=0.5, alpha=0.5)

# 색상 매핑
color_map = {'Starting Material': '#DC143C', 'Product': '#2E8B57',
             'Caffeine (QC)': '#FF8C00', 'Unknown': '#999'}

for _, row in assignments_df.iterrows():
    color = color_map.get(row['compound'], '#999')
    left, right = int(row['left_idx']), int(row['right_idx'])
    ax.fill_between(sample_mz[left:right+1], 0, corrected[left:right+1],
                    alpha=0.4, color=color)

    label = f"{row['compound']}\n{row['adduct']}" if row['compound'] != 'Unknown' else 'Unknown'
    ax.annotate(f"{row['mz']:.3f}\n{label}",
               xy=(row['mz'], row['corrected_intensity']),
               xytext=(0, 20), textcoords='offset points',
               ha='center', fontsize=7, fontweight='bold', color=color,
               arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.5, label=n) for n, c in color_map.items()]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title('Compound-Peak Assignment')
plt.tight_layout()
plt.show()

# 결과 테이블
assignments_df[['mz', 'compound', 'adduct', 'theoretical_mz', 'error_ppm', 'area', 'sn']].round(2)

### 4.3 최종 리포트 생성

In [ ]:
# ─── 최종 리포트 ───

print("="*60)
print("  Echo-TOF Analysis Report")
print("="*60)
print()

# 1. 피크 요약
print(f"[PEAKS] Detected Peaks: {len(peaks_df)}")
known = assignments_df[assignments_df['compound'] != 'Unknown']
unknown = assignments_df[assignments_df['compound'] == 'Unknown']
print(f"   Assigned: {len(known)} | Unknown: {len(unknown)}")
print()

# 2. 화합물별 요약
print("[COMPOUNDS] Compound Summary:")
for name in COMPOUNDS:
    comp_peaks = assignments_df[assignments_df['compound'] == name]
    if not comp_peaks.empty:
        best = comp_peaks.loc[comp_peaks['area'].idxmax()]
        print(f"   {name}:")
        print(f"     m/z = {best['mz']:.4f} ({best['adduct']})")
        print(f"     Area = {best['area']:.0f}, S/N = {best['sn']:.1f}")
        print(f"     Error = {best['error_ppm']:.1f} ppm")
    else:
        print(f"   {name}: NOT DETECTED")
print()

# 3. 플레이트 요약
print("[PLATE] Plate Summary (96-well):")
print(f"   Mean yield: {plate_df['yield_area_ratio'].mean():.1f}%")
print(f"   Median yield: {plate_df['yield_area_ratio'].median():.1f}%")
print(f"   Hit rate (>20%): {(plate_df['yield_area_ratio'] > 20).mean()*100:.0f}%")
print(f"   Best well: {plate_df.loc[plate_df['yield_area_ratio'].idxmax(), 'well']} ({plate_df['yield_area_ratio'].max():.1f}%)")
print(f"   Worst well: {plate_df.loc[plate_df['yield_area_ratio'].idxmin(), 'well']} ({plate_df['yield_area_ratio'].min():.1f}%)")
print()
print("✅ Analysis complete!")

---

## 다음 단계

- **실제 WIFF2 데이터 적용**: SCIEX OS에서 mzML로 변환 후 `load_mzml()` 사용
- **적분 파라미터 튜닝**: `PeakIntegrator(noise_factor=3.0)` 조정
- **커스텀 반응 조건**: `COMPOUNDS` 딕셔너리에 실제 화합물 추가
- **자동화**: Google Drive 연동으로 배치 분석 파이프라인 구축